In [1]:
import numpy as np
import pandas as pd

In [2]:
K = 3
INPUT_WINDOW = 22

RAW_DATA_PATH = "../data/raw/AAPL_1y_daily.csv"
LABELED_DATA_PATH = "../data/processed/AAPL_paper_labeled.csv"

print("K:", K)
print("Input window:", INPUT_WINDOW)

K: 3
Input window: 22


In [3]:
df = pd.read_csv(
    RAW_DATA_PATH,
    parse_dates=["Date"]
)

df = (
    df
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Shape:", df.shape)
print("Date range:")
print(
    df["Date"].min(),
    "→",
    df["Date"].max()
)

print("\nColumns:")
print(df.columns.tolist())

Shape: (251, 6)
Date range:
2025-08-18 00:00:00 → 2026-08-17 00:00:00

Columns:
['Date', 'Open', 'High', 'Low', 'Close', 'Volume']


In [4]:
assert "Date" in df.columns
assert "Close" in df.columns

assert df["Date"].is_monotonic_increasing
assert df["Date"].is_unique
assert df["Close"].notna().all()

print("Raw data validation: PASSED")

Raw data validation: PASSED


In [5]:
def create_paper_labels(
    data,
    k=3
):
    result = data.copy()

    closes = result["Close"].to_numpy(
        dtype=float
    )

    n = len(result)

    past_average = np.full(
        n,
        np.nan
    )

    future_average = np.full(
        n,
        np.nan
    )

    slope = np.full(
        n,
        np.nan
    )

    mu = np.full(
        n,
        np.nan
    )

    sigma = np.full(
        n,
        np.nan
    )

    paper_trend = np.full(
        n,
        None,
        dtype=object
    )

    # Valid target days:
    #
    # previous K-1 observations
    # current day
    # next K observations
    #
    # Therefore:
    # d = K-1 ... n-K-1

    for d in range(
        k - 1,
        n - k
    ):

        past_prices = closes[
            d - k + 1 : d + 1
        ]

        future_prices = closes[
            d + 1 : d + k + 1
        ]

        local_prices = closes[
            d - k + 1 : d + k + 1
        ]

        past_avg = np.mean(
            past_prices
        )

        future_avg = np.mean(
            future_prices
        )

        current_slope = (
            future_avg - past_avg
        )

        current_mu = np.mean(
            local_prices
        )

        # IMPORTANT:
        # Paper divides by 2K.
        # This is population standard deviation.
        current_sigma = np.sqrt(
            np.sum(
                (
                    local_prices
                    - current_mu
                ) ** 2
            )
            / (2 * k)
        )

        current_price = closes[d]

        # Paper's four-class rules
        if (
            current_price
            > current_mu + current_sigma
            and current_slope > 0
        ):
            trend = "Rise Plus"

        elif current_slope > 0:
            trend = "Rise"

        elif (
            current_price
            < current_mu - current_sigma
            and current_slope < 0
        ):
            trend = "Fall Plus"

        elif current_slope < 0:
            trend = "Fall"

        else:
            raise ValueError(
                f"Unable to label row {d}"
            )

        past_average[d] = past_avg
        future_average[d] = future_avg
        slope[d] = current_slope
        mu[d] = current_mu
        sigma[d] = current_sigma
        paper_trend[d] = trend

    result["past_average"] = past_average
    result["future_average"] = future_average
    result["slope"] = slope
    result["mu"] = mu
    result["sigma"] = sigma
    result["paper_trend"] = paper_trend

    return result

In [6]:
label_data = create_paper_labels(
    df,
    k=K
)

print(
    label_data[
        [
            "Date",
            "Close",
            "past_average",
            "future_average",
            "slope",
            "mu",
            "sigma",
            "paper_trend"
        ]
    ].head(10)
)

        Date       Close  past_average  future_average     slope          mu  \
0 2025-08-18  230.889999           NaN             NaN       NaN         NaN   
1 2025-08-19  230.559998           NaN             NaN       NaN         NaN   
2 2025-08-20  226.009995    229.153330      226.606664 -2.546666  227.879997   
3 2025-08-21  224.899994    227.156662      228.076665  0.920003  227.616664   
4 2025-08-22  227.759995    226.223328      228.986669  2.763341  227.604998   
5 2025-08-25  227.160004    226.606664      230.786667  4.180003  228.696665   
6 2025-08-26  229.309998    228.076665      231.730001  3.653336  229.903333   
7 2025-08-27  230.490005    228.986669      231.473333  2.486664  230.230001   
8 2025-08-28  232.559998    230.786667      233.443334  2.656667  232.115000   
9 2025-08-29  232.139999    231.730001      235.990000  4.260000  233.860001   

      sigma paper_trend  
0       NaN         NaN  
1       NaN         NaN  
2  2.204262        Fall  
3  1.902689    

In [7]:
label_data = label_data[
    label_data["paper_trend"].notna()
].copy()

label_data = (
    label_data
    .reset_index(drop=True)
)

print(
    "Labeled samples:",
    len(label_data)
)

print(
    "Date range:",
    label_data["Date"].min(),
    "→",
    label_data["Date"].max()
)

Labeled samples: 246
Date range: 2025-08-20 00:00:00 → 2026-08-12 00:00:00


In [8]:
example_index = 0

original_index = K - 1

local_prices = df.loc[
    original_index - K + 1 :
    original_index + K,
    "Close"
].to_numpy(
    dtype=float
)

manual_past_average = np.mean(
    local_prices[:K]
)

manual_future_average = np.mean(
    local_prices[K:]
)

manual_slope = (
    manual_future_average
    - manual_past_average
)

manual_mu = np.mean(
    local_prices
)

manual_sigma = np.sqrt(
    np.sum(
        (local_prices - manual_mu) ** 2
    )
    / (2 * K)
)

row = label_data.iloc[
    example_index
]

print(
    "Manual past average:",
    manual_past_average
)

print(
    "Stored past average:",
    row["past_average"]
)

print()

print(
    "Manual future average:",
    manual_future_average
)

print(
    "Stored future average:",
    row["future_average"]
)

print()

print(
    "Manual slope:",
    manual_slope
)

print(
    "Stored slope:",
    row["slope"]
)

print()

print(
    "Manual sigma:",
    manual_sigma
)

print(
    "Stored sigma:",
    row["sigma"]
)

assert np.isclose(
    manual_past_average,
    row["past_average"]
)

assert np.isclose(
    manual_future_average,
    row["future_average"]
)

assert np.isclose(
    manual_slope,
    row["slope"]
)

assert np.isclose(
    manual_sigma,
    row["sigma"]
)

print("\nPaper mathematics: PASSED")

Manual past average: 229.15333048502603
Stored past average: 229.15333048502603

Manual future average: 226.6066640218099
Stored future average: 226.6066640218099

Manual slope: -2.546666463216127
Stored slope: -2.546666463216127

Manual sigma: 2.2042623938681
Stored sigma: 2.2042623938681

Paper mathematics: PASSED


In [9]:
print("=" * 60)
print("PAPER FOUR-CLASS DISTRIBUTION")
print("=" * 60)

four_class_counts = (
    label_data["paper_trend"]
    .value_counts()
)

print(
    four_class_counts
)

print("\nPercentages:")

print(
    (
        label_data["paper_trend"]
        .value_counts(
            normalize=True
        )
        .mul(100)
        .round(2)
    )
)

PAPER FOUR-CLASS DISTRIBUTION
paper_trend
Rise         141
Fall          94
Fall Plus      7
Rise Plus      4
Name: count, dtype: int64

Percentages:
paper_trend
Rise         57.32
Fall         38.21
Fall Plus     2.85
Rise Plus     1.63
Name: proportion, dtype: float64


In [10]:
label_data["trend"] = (
    label_data["paper_trend"]
    .map({
        "Rise Plus": "Rise",
        "Rise": "Rise",
        "Fall Plus": "Fall",
        "Fall": "Fall"
    })
)

In [11]:
label_data["target"] = (
    label_data["trend"]
    .map({
        "Fall": 0,
        "Rise": 1
    })
)

In [12]:
assert label_data["trend"].notna().all()

assert label_data["target"].notna().all()

assert set(
    label_data["trend"].unique()
) == {
    "Rise",
    "Fall"
}

assert set(
    label_data["target"].unique()
) == {
    0,
    1
}

print("Binary label validation: PASSED")

Binary label validation: PASSED


In [13]:
print("=" * 60)
print("FINAL BINARY LABEL DISTRIBUTION")
print("=" * 60)

binary_counts = (
    label_data["trend"]
    .value_counts()
)

print(binary_counts)

print("\nPercentages:")

print(
    (
        label_data["trend"]
        .value_counts(
            normalize=True
        )
        .mul(100)
        .round(2)
    )
)

FINAL BINARY LABEL DISTRIBUTION
trend
Rise    145
Fall    101
Name: count, dtype: int64

Percentages:
trend
Rise    58.94
Fall    41.06
Name: proportion, dtype: float64


In [14]:
print(
    label_data[
        [
            "Date",
            "Close",
            "slope",
            "mu",
            "sigma",
            "paper_trend",
            "trend",
            "target"
        ]
    ].head(20)
)

         Date       Close     slope          mu     sigma paper_trend trend  \
0  2025-08-20  226.009995 -2.546666  227.879997  2.204262        Fall  Fall   
1  2025-08-21  224.899994  0.920003  227.616664  1.902689        Rise  Rise   
2  2025-08-22  227.759995  2.763341  227.604998  1.884738        Rise  Rise   
3  2025-08-25  227.160004  4.180003  228.696665  2.455300        Rise  Rise   
4  2025-08-26  229.309998  3.653336  229.903333  2.036195        Rise  Rise   
5  2025-08-27  230.490005  2.486664  230.230001  1.810892        Rise  Rise   
6  2025-08-28  232.559998  2.656667  232.115000  3.077590        Rise  Rise   
7  2025-08-29  232.139999  4.260000  233.860001  3.860988        Rise  Rise   
8  2025-09-02  229.720001  7.840001  235.393333  4.040808        Rise  Rise   
9  2025-09-03  238.470001  5.673335  236.280001  3.903149        Rise  Rise   
10 2025-09-04  239.779999  1.316671  236.648336  3.586518        Rise  Rise   
11 2025-09-05  239.690002 -6.306666  236.160001  4.5

In [15]:
label_data.to_csv(
    LABELED_DATA_PATH,
    index=False
)

print(
    "Saved corrected labels to:"
)

print(
    LABELED_DATA_PATH
)

Saved corrected labels to:
../data/processed/AAPL_paper_labeled.csv


In [16]:
print("=" * 60)
print("FINAL LABEL SANITY CHECK")
print("=" * 60)

print("Total labeled rows:", len(label_data))

print(
    "\nMissing values:"
)

print(
    label_data[
        [
            "past_average",
            "future_average",
            "slope",
            "mu",
            "sigma",
            "paper_trend",
            "trend",
            "target"
        ]
    ]
    .isna()
    .sum()
)

print("\nTarget distribution:")
print(
    label_data["target"]
    .value_counts()
    .sort_index()
)

print("\nDate order:")
print(
    "Chronological:",
    label_data["Date"].is_monotonic_increasing
)

print(
    "\nUnique dates:",
    label_data["Date"].nunique()
)

print(
    "Rows:",
    len(label_data)
)

FINAL LABEL SANITY CHECK
Total labeled rows: 246

Missing values:
past_average      0
future_average    0
slope             0
mu                0
sigma             0
paper_trend       0
trend             0
target            0
dtype: int64

Target distribution:
target
0    101
1    145
Name: count, dtype: int64

Date order:
Chronological: True

Unique dates: 246
Rows: 246
